In [1]:
# used the instructions of https://www.sparkcodehub.com/pyspark/use-cases/recommendation-systems
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col

# SparkSession (like the wpo)
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("AnimeRecommender") \
    .config("spark.ui.port", "4040") \
    .config("spark.hadoop.fs.defaultFS", "file:///") \
    .config("spark.driver.extraJavaOptions", "-Duser.name=admin") \
    .config("spark.driver.extraJavaOptions", "-Dsun.security.auth.login.config=C:/dev/null") \
    .getOrCreate()

In [2]:
#pre processed data
reviews_data = spark.read.csv("../data/preprocessed/reviews.csv", header=True, inferSchema=True)
anime_data = spark.read.csv("../data/preprocessed/animes.csv", header=True, inferSchema=True)

# The profiles need to be represented in numbers.
# Tried using cast like the intstructions but didn't work due to 
review_indexed = StringIndexer(inputCol="profile", outputCol="user_id")
reviews_data_indexed = review_indexed.fit(reviews_data).transform(reviews_data)

final_data = reviews_data_indexed.select(
    col("user_id").cast("integer"),
    col("anime_id").cast("integer"),
    col("score").cast("float")
)

final_data.show(7)

+-------+--------+-----+
|user_id|anime_id|score|
+-------+--------+-----+
|     32|   34096|  8.0|
|   1104|   34599| 10.0|
|   1825|   28891|  7.0|
|   3796|    2904|  9.0|
|   9589|    4181| 10.0|
|   9872|    2904| 10.0|
|    554|   16664|  6.0|
+-------+--------+-----+
only showing top 7 rows



In [3]:
## proof cast won't work
# test_data = [("---SnowFlake---",), ("--EYEPATCH--",), ("123",)]
# test_df = spark.createDataFrame(test_data, ["profile"])
# result_df = test_df.withColumn("user_id_test", col("profile").cast("integer"))
# result_df.show()

In [4]:
from pyspark.ml.recommendation import ALS

# Initialize ALS model
als = ALS(
    maxIter=10,              # Number of iterations
    regParam=0.1,            # Regularization parameter
    rank=10,                 # Number of latent factors
    userCol="user_id",       # User column
    itemCol="anime_id",      # Item column
    ratingCol="score",       # Rating column
    coldStartStrategy="drop" # Handle cold-start issues
)



In [5]:
# Train the model + Test
model = als.fit(final_data)

user_recs = model.recommendForAllUsers(10)
user_recs.limit(3).show(truncate=False)
user_recs.show(truncate=False)

+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                        |
+-------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|26     |[{4073, 14.558771}, {35267, 14.283359}, {30243, 13.845531}, {31189, 13.7748785}, {12875, 13.673424}, {28157, 13.384609}, {10720, 13.298077}, {2438, 12.859888}, {23213, 12.781411}, {33314, 12.773292}]|
|27     |[{30243, 13.937272}, {35267, 13.0581}, {10720, 12.833587}, {1558, 12.5837965}, {3992, 12.576638}, {12875, 12.153032}, {2955, 12.088473}, {28157, 11.853

In [8]:
##### Evaluating the model #######
from pyspark.ml.evaluation import RegressionEvaluator

train_df, test_df = final_data.randomSplit([0.8, 0.2], seed=42)

model = als.fit(train_df)

# Make predictions on test data
predictions = model.transform(test_df)

# Evaluate RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="score",
    predictionCol="prediction"
)
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")


Root Mean Squared Error (RMSE): 3.637604770940431


In [13]:
#### Hyper parameting ###
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder


param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [5, 10 , 15,  20]) \
    .addGrid(als.regParam, [0.01, 0.1, 0.5]) \
    .build()

# Set up cross-validator
crossval = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3
)

# Fit cross-validator
cv_model = crossval.fit(train_df)

# Best model
best_model = cv_model.bestModel

# Make predictions on test data
predictions = best_model.transform(test_df)

# Evaluate RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="score",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")

Root Mean Squared Error (RMSE): 2.4215551319266764


In [17]:
#### added bias (idk if it is the bias specified by the assistent it seems a bit different I think #####
from pyspark.sql import functions as F
# Calculate the mean rating per user
user_means = train_df.groupBy("user_id").agg(F.avg("score").alias("user_avg"))

# Join this average back into the training and testing datasets
train_with_means = train_df.join(user_means, on="user_id", how="left")
test_with_means = test_df.join(user_means, on="user_id", how="left") 

# Create the centered score: Actual Score - User Average
train_with_means = train_with_means.withColumn(
    "centered_score", 
    F.col("score") - F.col("user_avg")
)
train_with_means.select("user_id", "score", "user_avg", "centered_score").show(5)





# ALS with the bias for the users ( im tired) #########################################
als_bias = ALS(
    userCol="user_id",
    itemCol="anime_id",
    ratingCol="centered_score",   
    coldStartStrategy="drop"
)

param_grid = ParamGridBuilder() \
    .addGrid(als_bias.rank, [5, 10, 20]) \
    .addGrid(als_bias.regParam, [0.01, 0.1]) \
    .build()

crossval = CrossValidator(
    estimator=als_bias,
    estimatorParamMaps=param_grid,
    evaluator=evaluator, 
    numFolds=3
)



bias_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="centered_score",   
    predictionCol="prediction"
)

cv_model_biased = crossval.fit(train_with_means)
best_biased_model = cv_model_biased.bestModel


# Make predictions on the test data (which outputs a centered prediction)
predictions_with_bias = best_biased_model.transform(test_with_means)

# Add the user_avg back to get the real 1-10 prediction
final_predictions = predictions_with_bias.withColumn(
    "final_prediction", 
    F.col("prediction") + F.col("user_avg")
)

# Evaluate the FINAL RMSE on the original real 'score'
final_evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="score",               
    predictionCol="final_prediction" 
)

final_rmse = final_evaluator.evaluate(final_predictions)
print(f"Final RMSE (With Mean Centering): {final_rmse}")


#NOTE for tommorow: est ce que quand un user a pas beacoup de review est ce que ce serais pas plus inrteressant d'enlever le bias?
# RankingEvaluator, NDCG, Precision@3



+-------+-----+--------+--------------------+
|user_id|score|user_avg|      centered_score|
+-------+-----+--------+--------------------+
|      0|  5.0|     5.2|-0.20000000000000018|
|      0| 10.0|     5.2|                 4.8|
|      0|  8.0|     5.2|                 2.8|
|      0|  7.0|     5.2|  1.7999999999999998|
|      0|  7.0|     5.2|  1.7999999999999998|
+-------+-----+--------+--------------------+
only showing top 5 rows

Final RMSE (With Mean Centering): 2.2322676453824566
